# Couche sémantique v2 : modèle plus fort, index géré par une lib, hybride avec BM25

Suite de `semantic_layer.ipynb`, en gardant la même philosophie (autonome, tweakable,
concepts documentés) mais avec trois changements de fond :

1. **Un modèle d'embedding plus performant** — `paraphrase-multilingual-MiniLM-L12-v2`
   (v1, 384 dimensions, léger) est remplacé par `intfloat/multilingual-e5-large`
   (1024 dimensions, état de l'art multilingue open source), avec le piège qui va
   avec : les modèles E5 sont **asymétriques** (section 1).
2. **`numpy` seul ne gère plus la recherche.** v1 empilait tout dans une matrice et
   faisait un `matmul` exhaustif — exact, mais O(N) par requête. `hnswlib` (léger :
   un binaire C++ de quelques Mo, zéro service) construit un **index approximatif**
   (HNSW) qui génère des candidats en temps sous-linéaire ; on ne recalcule le score
   exact (`numpy`) que sur ce petit lot (section 2). `numpy` ne disparaît pas, son
   rôle change : de moteur de recherche à outil de rescoring.
3. **Recherche lexicale via OpenMetadata, pas réimplémentée.** OMD indexe déjà son
   catalogue dans Elasticsearch/OpenSearch avec du BM25 et des boosts de champ
   réglés (nom, description, colonnes...) — inutile de le refaire. On interroge son
   endpoint de recherche (`GET /v1/search/query`, le même que la barre de recherche
   de l'UI OMD) et on **fusionne** ce classement lexical avec le classement
   sémantique par *Reciprocal Rank Fusion* (section 4).

**Conséquence importante** : contrairement à v1, ce notebook a besoin d'une vraie
instance OpenMetadata pour que la partie hybride ait un sens (`OMD_HOST_PORT` /
`OMD_JWT_TOKEN` dans `.env`). La partie sémantique seule continue de tourner sur le
catalogue synthétique si OMD n'est pas joignable — `HybridSearch` se replie
automatiquement dessus (section 5).

**Nouvelles dépendances** : `hnswlib` (`poetry add hnswlib`). `httpx` est déjà tiré
par `openai` (aucun ajout).

In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import time
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("semantic_layer_v2")

# ============================================================================
# CONFIGURATION -- tout ce qui est tweakable est ici.
# ============================================================================

DATA_SOURCE = "synthetic"  # "synthetic" (aucune infra) | "omd" (catalogue OpenMetadata reel)

# Modele d'embedding : E5 est une famille ASYMETRIQUE (section 1). Alternatives
# multilingues plus legeres si le telechargement de 1024-dim est trop lourd :
# "intfloat/multilingual-e5-base" (768-dim, ~1.1 Go -> ~550 Mo) ou
# "intfloat/multilingual-e5-small" (384-dim, ~470 Mo) -- meme prefixage requis.
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
MODEL_CACHE_DIR = Path(
    os.environ.get("DOCMAKER_MODEL_CACHE", Path.home() / ".cache" / "docmaker" / "embeddings")
)

MEANING_WEIGHT = 0.5   # meme role qu'en v1 : poids du canal "sens" dans le score
INCLUDE_COLUMNS = True
INDEX_DIR = Path("build") / "semantic_layer_v2"

# hnswlib : parametres standards de la doc (M, ef_construction) -- augmenter M et
# ef_construction ameliore le recall au prix de la memoire/du temps de build ;
# ef_search au prix de la latence de recherche. Sans effet observable en dessous
# de quelques milliers de documents (l'index est alors quasi exact quoi qu'il arrive).
HNSW_M = 16
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 64
CANDIDATE_K = 20  # candidats generes par l'ANN avant filtrage/rescoring exact (section 2)

# Fusion semantique + lexical (section 4).
RRF_K = 60          # constante standard du papier RRF original, peu sensible
BM25_INDEX = "table_search_index"  # index de recherche OMD a interroger
BM25_TIMEOUT = 5.0

# Uniquement si DATA_SOURCE == "omd" :
OMD_SERVICE = "banking db"
OMD_SCHEMAS = {"ref", "stg", "ods", "dmt", "tec"}

## 0. Repris de v1 sans changement

Dictionnaire d'abréviations, modèle de données (`GlossaryEntry`/`RawTable`/
`RawColumn`/`Catalog`/`Document`), chargement du catalogue (synthétique ou OMD) et
composition des documents indexables : identiques à `semantic_layer.ipynb`, section
1 à 4. Redéfinis ici pour l'autonomie du notebook, sans commentaire supplémentaire —
voir v1 pour le détail et la justification de chaque choix.

In [ ]:
ABBREVIATIONS = {
    "CPT": "compte", "CLI": "client", "CRD": "credit", "MVT": "mouvement",
    "SLD": "solde", "OPE": "operation", "DOS": "dossier", "ADR": "adresse",
    "AGE": "agence", "ORG": "organisation", "PM": "personne morale",
    "PP": "personne physique",
    "MT": "montant", "NB": "nombre", "TX": "taux", "TXC": "taux de change",
    "TXI": "taux d'interet", "ENC": "encours", "EXP": "exposition", "TOT": "total",
    "RSQ": "risque",
    "DT": "date", "JR": "jour", "J": "journalier", "M": "mensuel", "MOIS": "mois",
    "DEB": "debut", "FIN": "fin", "OUV": "ouverture",
    "CD": "code", "ID": "identifiant", "LIB": "libelle", "TYP": "type", "NAT": "nature",
    "DEV": "devise", "CHF": "franc suisse", "PAY": "pays",
    "REF": "referentiel", "STG": "staging", "ODS": "donnees operationnelles",
    "DMT": "datamart", "DIM": "dimension", "F": "fait", "D": "dimension", "H": "historique",
    "BCK": "sauvegarde", "OLD": "ancien",
}


def expand_identifier(identifier: str, extra: dict[str, str] | None = None) -> str:
    tokens = identifier.upper().replace(".", "_").split("_")
    lookup = {**ABBREVIATIONS, **(extra or {})}
    return " ".join(lookup.get(token, token.lower()) for token in tokens)

In [ ]:
@dataclass
class GlossaryEntry:
    fqn: str
    name: str
    description: str = ""
    synonyms: list[str] = field(default_factory=list)

    def as_text(self) -> str:
        return " ".join(filter(None, [self.name, *self.synonyms, self.description]))


@dataclass
class RawColumn:
    name: str
    data_type: str = ""
    description: str = ""
    glossary_terms: list[str] = field(default_factory=list)
    tags: list[str] = field(default_factory=list)


@dataclass
class RawTable:
    fqn: str
    name: str
    schema: str
    description: str = ""
    glossary_terms: list[str] = field(default_factory=list)
    tags: list[str] = field(default_factory=list)
    columns: list[RawColumn] = field(default_factory=list)


@dataclass
class Catalog:
    tables: list[RawTable]
    glossary: dict[str, GlossaryEntry]

    def synonym_map(self) -> dict[str, str]:
        mapping: dict[str, str] = {}
        for entry in self.glossary.values():
            for synonym in entry.synonyms:
                token = synonym.strip().upper()
                if token and " " not in token:
                    mapping[token] = entry.name.lower()
        return mapping


@dataclass
class Document:
    fqn: str
    entity_type: str  # "table" | "column"
    name: str
    schema: str
    identity_text: str
    meaning_text: str = ""
    tags: list[str] = field(default_factory=list)
    has_description: bool = False

    @property
    def has_meaning(self) -> bool:
        return bool(self.meaning_text.strip())

    def to_dict(self) -> dict:
        return {
            "fqn": self.fqn, "entity_type": self.entity_type, "name": self.name,
            "schema": self.schema, "identity_text": self.identity_text,
            "meaning_text": self.meaning_text, "tags": self.tags,
            "has_description": self.has_description,
        }

    @classmethod
    def from_dict(cls, data: dict) -> "Document":
        return cls(**data)


@dataclass
class SearchHit:
    score: float
    document: Document


def _passes(
    doc: Document, entity_type: str | None = None, schema: str | None = None,
    tags: set[str] | None = None, described_only: bool = False,
) -> bool:
    """Predicat de filtre par facette, partage par le moteur semantique (section 2)
    et par la fusion hybride (section 5) -- un seul endroit qui sait ce qu'un filtre
    signifie.
    """
    if entity_type and doc.entity_type != entity_type:
        return False
    if schema and doc.schema.lower() != schema.lower():
        return False
    if tags and not tags.intersection(doc.tags):
        return False
    if described_only and not doc.has_description:
        return False
    return True

In [ ]:
def load_synthetic_catalog(enriched: bool = True) -> Catalog:
    def col(name, data_type="VARCHAR2", description="", terms=None):
        return RawColumn(
            name=name, data_type=data_type,
            description=description if enriched else "",
            glossary_terms=(terms or []) if enriched else [],
        )

    tables = [
        RawTable(
            fqn="DMT.DMT_CPT_MVT_J", name="DMT_CPT_MVT_J", schema="dmt",
            description=(
                "Mouvements comptables journaliers par compte : une ligne par "
                "operation debitrice ou creditrice." if enriched else ""
            ),
            columns=[
                col("id_mvt", "NUMBER"), col("id_compte", "NUMBER"), col("dt_mvt", "DATE"),
                col("cd_typ_ope", "VARCHAR2",
                    "Code du type d'operation : debit, credit, virement, prelevement."),
                col("mt_mvt", "NUMBER", "Montant de l'operation dans la devise d'origine."),
            ],
        ),
        RawTable(
            fqn="DMT.DMT_CPT_SLD_J", name="DMT_CPT_SLD_J", schema="dmt",
            description="Solde de fin de journee par compte, en francs suisses." if enriched else "",
            columns=[col("id_compte", "NUMBER"), col("dt_jour", "DATE"), col("mt_sld_chf", "NUMBER")],
        ),
        RawTable(
            fqn="ODS.ODS_D_CLI_ADR", name="ODS_D_CLI_ADR", schema="ods",
            description=(
                "Adresses postales des clients, historisees par periode de validite."
                if enriched else ""
            ),
            columns=[
                col("id_client", "NUMBER"),
                col("rue", "VARCHAR2", "Libelle de voie de l'adresse postale."),
                col("ville", "VARCHAR2"), col("dt_deb_val", "DATE"),
            ],
        ),
        RawTable(
            fqn="DMT.DMT_F_CRD_ENC_M", name="DMT_F_CRD_ENC_M", schema="dmt",
            description="Encours de credit mensuels par dossier et par agence." if enriched else "",
            columns=[
                col("id_dossier", "NUMBER"), col("dt_fin_mois", "DATE"),
                col("mt_crd_restant", "NUMBER",
                    "Capital restant du sur le dossier de credit a la fin du mois.",
                    terms=["Banque.Encours"]),
            ],
        ),
        RawTable(
            fqn="REF.V_REF_FIN_TXC_CHF", name="V_REF_FIN_TXC_CHF", schema="ref",
            description=(
                "Cours de conversion quotidiens des devises vers le franc suisse."
                if enriched else ""
            ),
            columns=[col("devise", "VARCHAR2"), col("dt_jour", "DATE"), col("tx_chf", "NUMBER")],
        ),
        RawTable(
            fqn="ODS.ODS_F_CPT_MVT_BCK_2019", name="ODS_F_CPT_MVT_BCK_2019", schema="ods",
            columns=[col("id_mvt", "NUMBER"), col("mt_mvt", "NUMBER")],
        ),
    ]

    glossary: dict[str, GlossaryEntry] = {}
    if enriched:
        entries = [
            GlossaryEntry("Banque.Solde disponible", "Solde disponible",
                          "Montant utilisable immediatement par le client.",
                          ["SLD", "avoir disponible"]),
            GlossaryEntry("Banque.Mouvement", "Mouvement",
                          "Ecriture comptable passee sur un compte.",
                          ["MVT", "operation", "transaction"]),
            GlossaryEntry("Banque.Encours", "Encours",
                          "Capital restant du sur un credit a une date donnee.", ["ENC"]),
            GlossaryEntry("Banque.Taux de change", "Taux de change",
                          "Cours de conversion d'une devise vers une autre.", ["TXC", "TX"]),
        ]
        glossary = {e.fqn: e for e in entries}

    return Catalog(tables=tables, glossary=glossary)


def load_omd_catalog(service: str, schemas: set[str]) -> Catalog:
    from metadata.generated.schema.entity.data.glossaryTerm import GlossaryTerm
    from metadata.generated.schema.entity.data.table import Table as OMDTable
    from metadata.generated.schema.entity.services.connections.metadata.openMetadataConnection import (
        OpenMetadataConnection,
    )
    from metadata.generated.schema.security.client.openMetadataJWTClientConfig import (
        OpenMetadataJWTClientConfig,
    )
    from metadata.generated.schema.type.tagLabel import TagSource
    from metadata.ingestion.ometa.ometa_api import OpenMetadata

    connection = OpenMetadataConnection(
        hostPort=os.environ["OMD_HOST_PORT"],
        securityConfig=OpenMetadataJWTClientConfig(jwtToken=os.environ["OMD_JWT_TOKEN"]),
    )
    client = OpenMetadata(connection)

    glossary: dict[str, GlossaryEntry] = {}
    for term in client.list_all_entities(entity=GlossaryTerm, fields=["relatedTerms"]):
        fqn = term.fullyQualifiedName.root if term.fullyQualifiedName else term.name.root
        glossary[fqn] = GlossaryEntry(
            fqn=fqn, name=(term.displayName or term.name.root),
            description=term.description.root if term.description else "",
            synonyms=[s.root if hasattr(s, "root") else str(s) for s in (term.synonyms or [])],
        )

    def split_tags(tags):
        glossary_fqns, classification_fqns = [], []
        for label in tags or []:
            fqn = label.tagFQN.root if hasattr(label.tagFQN, "root") else str(label.tagFQN)
            (glossary_fqns if label.source == TagSource.Glossary else classification_fqns).append(fqn)
        return glossary_fqns, classification_fqns

    def adapt_column(c) -> RawColumn:
        glossary_fqns, classification_fqns = split_tags(c.tags)
        return RawColumn(
            name=c.name.root, data_type=c.dataTypeDisplay or "",
            description=c.description.root if c.description else "",
            glossary_terms=glossary_fqns, tags=classification_fqns,
        )

    tables: list[RawTable] = []
    for t in client.list_all_entities(
        entity=OMDTable, fields=["columns", "tags", "owners"], params={"service": service}
    ):
        schema = t.databaseSchema.name if t.databaseSchema else ""
        if schemas and schema.lower() not in schemas:
            continue
        glossary_fqns, classification_fqns = split_tags(t.tags)
        tables.append(RawTable(
            fqn=t.fullyQualifiedName.root, name=t.name.root, schema=schema,
            description=t.description.root if t.description else "",
            glossary_terms=glossary_fqns, tags=classification_fqns,
            columns=[adapt_column(c) for c in (t.columns or [])],
        ))
    return Catalog(tables=tables, glossary=glossary)


def get_catalog(source: str = DATA_SOURCE) -> Catalog:
    if source == "synthetic":
        return load_synthetic_catalog(enriched=True)
    if source == "omd":
        return load_omd_catalog(OMD_SERVICE, OMD_SCHEMAS)
    raise ValueError(f"DATA_SOURCE inconnu : {source!r}")


def _glossary_text(fqns: list[str], glossary: dict[str, GlossaryEntry]) -> str:
    return " ; ".join(glossary[fqn].as_text() for fqn in fqns if fqn in glossary)


def _join(*parts: str) -> str:
    return "\n".join(p.strip() for p in parts if p and p.strip())


def build_documents(catalog: Catalog, include_columns: bool = INCLUDE_COLUMNS) -> list[Document]:
    synonyms = catalog.synonym_map()
    documents: list[Document] = []
    for table in catalog.tables:
        documents.append(Document(
            fqn=table.fqn, entity_type="table", name=table.name, schema=table.schema,
            identity_text=_join(
                f"table {expand_identifier(table.name, synonyms)}",
                "colonnes : " + ", ".join(expand_identifier(c.name, synonyms) for c in table.columns),
            ),
            meaning_text=_join(table.description, _glossary_text(table.glossary_terms, catalog.glossary)),
            tags=table.tags, has_description=bool(table.description),
        ))
        if not include_columns:
            continue
        for column in table.columns:
            documents.append(Document(
                fqn=f"{table.fqn}.{column.name}", entity_type="column", name=column.name,
                schema=table.schema,
                identity_text=_join(
                    f"colonne {expand_identifier(column.name, synonyms)} "
                    f"de la table {expand_identifier(table.name, synonyms)}",
                    f"type {column.data_type}",
                ),
                meaning_text=_join(column.description, _glossary_text(column.glossary_terms, catalog.glossary)),
                tags=column.tags, has_description=bool(column.description),
            ))
    return documents


catalog = get_catalog()
documents = build_documents(catalog)
print(f"{len(catalog.tables)} table(s), {len(documents)} document(s) indexables")

## 1. Un modèle plus fort — et le piège des modèles asymétriques

`intfloat/multilingual-e5-large` est entraîné avec une **instruction différente selon
le rôle du texte** : `"query: {texte}"` pour ce qu'on cherche, `"passage: {texte}"`
pour ce qui est indexé. Oublier ce préfixe ne fait pas planter le code — les vecteurs
sortent quand même — mais dégrade silencieusement la qualité du classement, puisque
le modèle n'a jamais vu ce texte sans préfixe à l'entraînement. C'est le genre de
détail qui ne se voit pas en relisant le code, seulement en mesurant (section 6).

Tous les modèles n'ont pas cette convention (la famille `sentence-transformers/*` de
v1 n'en a pas) : `_PASSAGE_PREFIX`/`_QUERY_PREFIX` sont donc vides par défaut, et
appliqués uniquement pour la famille E5 détectée par le nom du modèle.

In [ ]:
def _is_e5_model(model_name: str) -> bool:
    return "e5" in model_name.lower()


_PASSAGE_PREFIX = "passage: " if _is_e5_model(EMBEDDING_MODEL) else ""
_QUERY_PREFIX = "query: " if _is_e5_model(EMBEDDING_MODEL) else ""


def _passage(text: str) -> str:
    return _PASSAGE_PREFIX + text


def _query(text: str) -> str:
    return _QUERY_PREFIX + text


def _embedder(model_name: str):
    """Import tardif (comme en v1). Verifie que le modele demande existe reellement
    dans cette version de fastembed avant de lancer un telechargement qui echouerait
    a la moitie -- et liste des alternatives multilingues si ce n'est pas le cas.
    """
    from fastembed import TextEmbedding

    available = {m["model"] for m in TextEmbedding.list_supported_models()}
    if model_name not in available:
        multilingual = sorted(m for m in available if "multilingual" in m or "e5" in m)
        raise ValueError(
            f"{model_name!r} n'est pas supporte par cette version de fastembed. "
            f"Options multilingues disponibles : {multilingual}"
        )
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return TextEmbedding(model_name, cache_dir=str(MODEL_CACHE_DIR))

## 2. hnswlib : générer des candidats à l'échelle, puis rescorer exactement

**HNSW** (Hierarchical Navigable Small World) construit un graphe où chaque vecteur
est relié à ses voisins approximatifs ; la recherche descend ce graphe en `O(log N)`
au lieu de comparer à tous les documents. `hnswlib` l'implémente en C++ avec une API
minimale (`add_items`, `knn_query`), sans service ni fichier de config.

**Le pattern qui compte ici** : HNSW donne une liste de candidats *approximative*
(bonne mais pas garantie exacte), pas un score de confiance absolu. Pour garder la
formule calibrée de v1 (`max(identité, mélange)`, calée sur le golden set), on ne
fait confiance à HNSW que pour *proposer* des candidats — le score final de chaque
candidat est recalculé exactement (produit scalaire numpy, comme en v1), juste sur
ce petit lot au lieu de tout le corpus. Deux index HNSW distincts (un par canal,
comme les deux matrices de v1) : `identity_store` sur tous les documents,
`meaning_store` uniquement sur les documents qui ont du texte de sens (un vecteur nul
n'a pas de direction, HNSW n'en veut pas).

**Limite assumée** (v1 ne l'avait pas, exhaustif par construction) : le filtre par
facette s'applique *après* la génération de candidats. Un filtre restrictif combiné à
un `CANDIDATE_K` trop petit peut renvoyer moins de `top_k` résultats même s'il en
existe davantage plus loin dans le corpus — augmenter `CANDIDATE_K` élargit le filet
au prix de la latence.

In [ ]:
try:
    import hnswlib
except ImportError as e:
    raise ImportError(
        "hnswlib n'est pas installe dans cet environnement. "
        "Poetry : `poetry add hnswlib`. Sinon : `pip install hnswlib`."
    ) from e


def _normalize(matrix: np.ndarray) -> np.ndarray:
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


@dataclass
class HnswVectorStore:
    """Index HNSW + la table `ids` qui retrouve la position originale du document
    (necessaire car `meaning_store` n'indexe qu'un sous-ensemble des documents :
    l'id interne hnswlib [0..n) ne correspond pas a la position dans `documents`).
    """

    index: "hnswlib.Index"
    ids: list[int]

    @classmethod
    def build(
        cls, vectors: np.ndarray, ids: list[int],
        M: int = HNSW_M, ef_construction: int = HNSW_EF_CONSTRUCTION,
    ) -> "HnswVectorStore":
        index = hnswlib.Index(space="cosine", dim=vectors.shape[1])
        index.init_index(max_elements=len(vectors), M=M, ef_construction=ef_construction)
        index.add_items(vectors, list(range(len(vectors))))
        index.set_ef(max(HNSW_EF_SEARCH, len(vectors)))
        return cls(index=index, ids=list(ids))

    def search(self, vector: np.ndarray, k: int) -> list[int]:
        """Renvoie les positions originales (dans `documents`) des k candidats les
        plus proches -- pas leur score : le score exact est recalcule en section 2.
        """
        k = min(k, self.index.get_current_count())
        if k == 0:
            return []
        labels, _ = self.index.knn_query(vector.reshape(1, -1), k=k)
        return [self.ids[i] for i in labels[0]]

    def save(self, path: Path) -> None:
        self.index.save_index(str(path))
        (path.with_suffix(".ids.json")).write_text(json.dumps(self.ids))

    @classmethod
    def load(cls, path: Path, dim: int) -> "HnswVectorStore":
        ids = json.loads(path.with_suffix(".ids.json").read_text())
        index = hnswlib.Index(space="cosine", dim=dim)
        index.load_index(str(path), max_elements=len(ids))
        index.set_ef(max(HNSW_EF_SEARCH, len(ids)))
        return cls(index=index, ids=ids)

In [ ]:
def corpus_hash(documents: list[Document]) -> str:
    digest = hashlib.sha256()
    for d in documents:
        digest.update(d.fqn.encode("utf-8"))
        digest.update(d.identity_text.encode("utf-8"))
        digest.update(d.meaning_text.encode("utf-8"))
    return digest.hexdigest()[:16]


@dataclass
class SemanticIndexV2:
    """Meme contrat que `SemanticIndex` de v1 (`.search(question, top_k, **filtres)`
    renvoie des `SearchHit`), moteur different : generation de candidats par
    `hnswlib`, rescoring exact par produit scalaire sur ce lot seulement.
    """

    documents: list[Document]
    identity_store: HnswVectorStore
    meaning_store: HnswVectorStore | None  # None si aucun document n'a de texte de sens
    identity_vectors: np.ndarray  # gardees pour le rescoring exact
    meaning_vectors: dict[int, np.ndarray]  # position document -> vecteur sens (documentes seulement)
    model_name: str = EMBEDDING_MODEL
    meaning_weight: float = MEANING_WEIGHT
    candidate_k: int = CANDIDATE_K

    @classmethod
    def build(
        cls, documents: list[Document], model_name: str = EMBEDDING_MODEL,
        meaning_weight: float = MEANING_WEIGHT, candidate_k: int = CANDIDATE_K,
    ) -> "SemanticIndexV2":
        model = _embedder(model_name)
        identity_vectors = _normalize(
            np.array(list(model.embed([_passage(d.identity_text) for d in documents])))
        )
        documented = [i for i, d in enumerate(documents) if d.has_meaning]
        meaning_vectors: dict[int, np.ndarray] = {}
        meaning_store = None
        if documented:
            meaning_matrix = _normalize(
                np.array(list(model.embed([_passage(documents[i].meaning_text) for i in documented])))
            )
            meaning_vectors = dict(zip(documented, meaning_matrix, strict=True))
            meaning_store = HnswVectorStore.build(meaning_matrix, ids=documented)

        identity_store = HnswVectorStore.build(identity_vectors, ids=list(range(len(documents))))
        log.info(
            "index v2 construit : %d document(s), %d document(s) documente(s), dim=%d",
            len(documents), len(documented), identity_vectors.shape[1],
        )
        return cls(
            documents=documents, identity_store=identity_store, meaning_store=meaning_store,
            identity_vectors=identity_vectors, meaning_vectors=meaning_vectors,
            model_name=model_name, meaning_weight=meaning_weight, candidate_k=candidate_k,
        )

    def save(self, directory: Path = INDEX_DIR) -> Path:
        directory.mkdir(parents=True, exist_ok=True)
        self.identity_store.save(directory / "identity.hnsw")
        if self.meaning_store is not None:
            self.meaning_store.save(directory / "meaning.hnsw")
        np.save(directory / "identity_vectors.npy", self.identity_vectors)
        with (directory / "documents.jsonl").open("w", encoding="utf-8") as fh:
            for d in self.documents:
                fh.write(json.dumps(d.to_dict(), ensure_ascii=False) + "\n")
        (directory / "meta.json").write_text(json.dumps({
            "model": self.model_name, "meaning_weight": self.meaning_weight,
            "dimensions": int(self.identity_vectors.shape[1]),
            "has_meaning_store": self.meaning_store is not None,
            "corpus_hash": corpus_hash(self.documents),
            "built_at": datetime.now(UTC).isoformat(),
        }, indent=2), encoding="utf-8")
        return directory

    def is_stale(self, documents: list[Document]) -> bool:
        return corpus_hash(documents) != corpus_hash(self.documents)

    def search(
        self, question: str, top_k: int = 5, entity_type: str | None = None,
        schema: str | None = None, tags: set[str] | None = None,
        described_only: bool = False,
    ) -> list[SearchHit]:
        model = _embedder(self.model_name)
        vector = next(iter(model.embed([_query(question)])))
        vector = vector / np.linalg.norm(vector)

        candidates = set(self.identity_store.search(vector, self.candidate_k))
        if self.meaning_store is not None:
            candidates |= set(self.meaning_store.search(vector, self.candidate_k))

        hits: list[SearchHit] = []
        for pos in candidates:
            doc = self.documents[pos]
            if not _passes(doc, entity_type, schema, tags, described_only):
                continue
            identity_score = float(self.identity_vectors[pos] @ vector)
            if doc.has_meaning:
                meaning_score = float(self.meaning_vectors[pos] @ vector)
                blended = (1 - self.meaning_weight) * identity_score + self.meaning_weight * meaning_score
                score = max(identity_score, blended)
            else:
                score = identity_score
            hits.append(SearchHit(score=score, document=doc))

        hits.sort(key=lambda h: -h.score)
        return hits[:top_k]

In [ ]:
start = time.perf_counter()
semantic_v2 = SemanticIndexV2.build(documents)
semantic_v2.save()
print(f"index v2 construit en {time.perf_counter() - start:.1f}s -> {INDEX_DIR}/")


def montre(question: str, engine, k: int = 3, **filtres) -> None:
    print(f"\nQ: {question}")
    for hit in engine.search(question, top_k=k, **filtres):
        doc = hit.document
        marque = " +desc" if doc.has_description else ""
        print(f"   {hit.score:.3f}  {doc.entity_type:6} {doc.name}{marque}")


for question in [
    "solde journalier d'un compte", "mouvements debiteurs du mois",
    "capital restant du sur un credit", "adresse du client",
]:
    montre(question, semantic_v2, entity_type="table")

## 3. Recherche lexicale : l'endpoint BM25 natif d'OpenMetadata

OMD indexe son catalogue dans Elasticsearch/OpenSearch et expose `GET
/v1/search/query` — le même endpoint que la barre de recherche de son UI, avec les
boosts de champ déjà réglés côté OMD (nom, description, colonnes...). On l'appelle
tel quel, sans rien réimplémenter :

```
GET {OMD_HOST_PORT}/v1/search/query?q=...&index=table_search_index&size=...
    &deleted=false&sort_field=_score&sort_order=desc
```

La réponse a la forme Elasticsearch classique : `hits.hits[]`, chaque élément portant
`_score` (le score BM25, non borné) et `_source.fullyQualifiedName`. `sort_field`
doit être explicitement `_score` — le tri par défaut de cet endpoint est alphabétique
sur `fullyQualifiedName`, pas par pertinence.

`try_connect` ne lève jamais : c'est ce qui permet à `HybridSearch` (section 5) de se
replier proprement sur le sémantique seul si OMD est absent.

In [ ]:
import httpx  # deja tire par `openai` (aucune dependance ajoutee)


@dataclass
class Hit:
    fqn: str
    score: float
    name: str = ""


class BM25Backend:
    def __init__(self, host_port: str, jwt_token: str, index: str = BM25_INDEX):
        self._client = httpx.Client(
            base_url=host_port, headers={"Authorization": f"Bearer {jwt_token}"},
            timeout=BM25_TIMEOUT,
        )
        self._index = index
        # ping immediat : on veut echouer a la construction, pas au milieu d'une recherche.
        self._client.get("/v1/search/query", params={"q": "*", "index": index, "size": 1}).raise_for_status()
        log.info("BM25 (OMD) disponible sur %s", host_port)

    def search(self, query: str, top_k: int = 10) -> list[Hit]:
        response = self._client.get("/v1/search/query", params={
            "q": query, "index": self._index, "from": 0, "size": top_k,
            "deleted": "false", "sort_field": "_score", "sort_order": "desc",
        })
        response.raise_for_status()
        hits = response.json().get("hits", {}).get("hits", [])
        return [
            Hit(fqn=h["_source"]["fullyQualifiedName"], score=h["_score"], name=h["_source"].get("name", ""))
            for h in hits
        ]

    @classmethod
    def try_connect(cls, host_port: str | None, jwt_token: str | None) -> "BM25Backend | None":
        if not host_port or not jwt_token:
            log.warning("OMD_HOST_PORT/OMD_JWT_TOKEN absent(s) : BM25 desactive, semantique seul.")
            return None
        try:
            return cls(host_port, jwt_token)
        except Exception as e:  # noqa: BLE001 -- jamais d'exception cote appelant
            log.warning("OMD injoignable (%s) : BM25 desactive, semantique seul.", e)
            return None


bm25 = BM25Backend.try_connect(os.environ.get("OMD_HOST_PORT"), os.environ.get("OMD_JWT_TOKEN"))
if bm25 is not None:
    print(bm25.search("solde journalier d'un compte"))
else:
    print("BM25 indisponible dans cet environnement -- attendu si OMD n'est pas lance localement.")

## 4. Fusionner : Reciprocal Rank Fusion

Un score cosinus (borné `[-1, 1]`) et un score BM25 (non borné, dépend du corpus et
de la requête) ne vivent pas sur la même échelle — les mélanger directement, comme le
fait `MEANING_WEIGHT` en section 2, serait arbitraire : ça marchait en v1 parce que
les deux canaux partagent le même espace vectoriel, ce qui n'est plus le cas ici.

**RRF** contourne le problème en ne regardant que le **rang**, jamais le score brut :
chaque document reçoit `1 / (k + rang)` dans chaque classement où il apparaît, les
contributions s'additionnent. Un document bien classé par les deux moteurs remonte
mécaniquement ; un document que seul un moteur trouve garde une contribution modeste
mais non nulle. `k` (traditionnellement 60, papier original de Cormack et al.) amortit
l'écart entre la 1ère et la 2e position — peu sensible en pratique.

In [ ]:
def reciprocal_rank_fusion(rankings: list[list[str]], k: int = RRF_K) -> dict[str, float]:
    """`rankings` : une liste de classements (chacun une liste de FQN, du meilleur au
    moins bon). Renvoie un score de fusion par FQN, pas encore trie.
    """
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, fqn in enumerate(ranking, start=1):
            scores[fqn] = scores.get(fqn, 0.0) + 1.0 / (k + rank)
    return scores

## 5. `HybridSearch` : une façade simple, difficile à utiliser de travers

Une seule méthode publique, `.search(query, top_k, **filtres)`. Les filtres par
facette (schéma, type d'entité, tags) s'appliquent aux deux classements avant fusion,
via le même `_passes` que `SemanticIndexV2` — sans ça, un résultat BM25 hors filtre
pourrait remonter dans la fusion. Toute panne du moteur lexical (OMD éteint, timeout,
erreur réseau ponctuelle) est absorbée : la recherche continue en sémantique seul au
lieu de faire planter l'appelant.

In [ ]:
@dataclass
class HybridHit:
    fqn: str
    rrf_score: float
    document: Document | None


class HybridSearch:
    def __init__(self, semantic: SemanticIndexV2, lexical: "BM25Backend | None"):
        self.semantic = semantic
        self.lexical = lexical
        self._by_fqn = {d.fqn: d for d in semantic.documents}

    def search(self, query: str, top_k: int = 5, candidate_k: int = CANDIDATE_K, **filters) -> list[HybridHit]:
        semantic_hits = self.semantic.search(query, top_k=candidate_k, **filters)
        rankings = [[h.document.fqn for h in semantic_hits]]

        if self.lexical is not None:
            try:
                lexical_hits = self.lexical.search(query, top_k=candidate_k)
                rankings.append([
                    h.fqn for h in lexical_hits
                    if (doc := self._by_fqn.get(h.fqn)) and _passes(doc, **filters)
                ])
            except Exception as e:  # noqa: BLE001 -- une panne ponctuelle ne casse pas la recherche
                log.warning("BM25 indisponible pour cette requete (%s) : semantique seul.", e)

        fused = reciprocal_rank_fusion(rankings)
        best = sorted(fused.items(), key=lambda kv: -kv[1])[:top_k]
        return [HybridHit(fqn=fqn, rrf_score=score, document=self._by_fqn.get(fqn)) for fqn, score in best]


hybrid = HybridSearch(semantic_v2, bm25)


def montre_hybride(question: str, k: int = 3, **filtres) -> None:
    print(f"\nQ: {question}")
    for hit in hybrid.search(question, top_k=k, **filtres):
        doc = hit.document
        marque = " +desc" if doc and doc.has_description else ""
        nom = doc.name if doc else hit.fqn
        print(f"   rrf={hit.rrf_score:.4f}  {nom}{marque}")


for question in ["solde journalier d'un compte", "capital restant du sur un credit"]:
    montre_hybride(question, entity_type="table")

## 6. Mesurer : hybride vs sémantique seul

Même golden set qu'en v1. **Mise en garde honnête** : le golden set vise les noms du
catalogue *synthétique* ; si `bm25` est branché sur une instance OMD qui contient un
catalogue *différent* (le cas si `OMD_HOST_PORT` pointe vers votre vraie instance
sans y avoir chargé ce catalogue de démo), BM25 ne trouvera simplement rien de
pertinent — la fusion RRF dégénère alors vers le classement sémantique seul, et les
deux mesures ci-dessous seront identiques. C'est le comportement *correct* de la
fusion (pas un bug), mais ça veut dire que la mesure n'est probante qu'avec un OMD
dont le catalogue correspond réellement au golden set utilisé.

In [ ]:
GOLDEN = {
    "solde journalier d'un compte": "DMT_CPT_SLD_J",
    "mouvements debiteurs du mois": "DMT_CPT_MVT_J",
    "capital restant du sur un credit": "DMT_F_CRD_ENC_M",
    "adresse du client": "ODS_D_CLI_ADR",
    "taux de change vers le franc suisse": "V_REF_FIN_TXC_CHF",
}


def recall_at_k(top_names_fn, k: int = 3) -> None:
    trouves = []
    for question, attendu in GOLDEN.items():
        noms = top_names_fn(question, k)
        rang = noms.index(attendu) + 1 if attendu in noms else None
        trouves.append(rang is not None)
        etat = f"rang {rang}" if rang else "RATE"
        top1 = noms[0] if noms else "(aucun)"
        print(f"   {question:38} {etat:8} top1={top1}")
    print(f"   recall@{k} = {sum(trouves)}/{len(GOLDEN)}\n")


def semantic_names(question: str, k: int) -> list[str]:
    return [h.document.name for h in semantic_v2.search(question, top_k=k, entity_type="table")]


def hybrid_names(question: str, k: int) -> list[str]:
    return [h.document.name for h in hybrid.search(question, top_k=k, entity_type="table") if h.document]


print("-- semantique seul (v2 : e5-large + hnswlib)")
recall_at_k(semantic_names)
print("-- hybride (semantique + BM25 OMD, fusion RRF)")
recall_at_k(hybrid_names)

## 7. Quoi tweaker

| Variable                             | Effet                                                                 |
| ------------------------------------- | ---------------------------------------------------------------------- |
| `EMBEDDING_MODEL`                     | tout modèle fastembed ; si "e5" dans le nom, le préfixage s'active seul |
| `HNSW_M` / `HNSW_EF_CONSTRUCTION`     | qualité de l'index au prix du temps de build (defaults hnswlib usuels) |
| `HNSW_EF_SEARCH` / `CANDIDATE_K`      | recall de la recherche au prix de la latence                          |
| `RRF_K`                               | amortissement de la fusion — peu sensible, 60 est le défaut usuel      |
| `BM25_INDEX`                          | cibler un autre index OMD (`dashboard_search_index`, `all`, ...)       |
| `DATA_SOURCE`                         | `"synthetic"` (démo) ↔ `"omd"` (catalogue réel — nécessaire pour que la mesure de la section 6 soit probante) |

**Ce qui a changé de fond par rapport à v1** : `numpy` reste le calculateur de score
(le `@` exact), mais n'est plus le moteur de recherche — ce rôle revient à `hnswlib`.
Le calibrage `max(identité, mélange)` de v1 est *réutilisé tel quel* ici, juste
appliqué à un lot de candidats plus petit : la théorie ne change pas, seulement
l'échelle à laquelle elle s'exécute.